## <small>
Copyright (c) 2017-21 Andrew Glassner

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
</small>



# Deep Learning: A Visual Approach
## by Andrew Glassner, https://glassner.com
### Order: https://nostarch.com/deep-learning-visual-approach
### GitHub: https://github.com/blueberrymusic
------

### What's in this notebook

This notebook is provided as a “behind-the-scenes” look at code used to make some of the figures in this chapter. It is cleaned up a bit from the original code that I hacked together, and is only lightly commented. I wrote the code to be easy to interpret and understand, even for those who are new to Python. I tried never to be clever or even more efficient at the cost of being harder to understand. The code is in Python3, using the versions of libraries as of April 2021.

This notebook may contain additional code to create models and images not in the book. That material is included here to demonstrate additional techniques.

Note that I've included the output cells in this saved notebook, but Jupyter doesn't save the variables or data that were used to generate them. To recreate any cell's output, evaluate all the cells from the start up to that cell. A convenient way to experiment is to first choose "Restart & Run All" from the Kernel menu, so that everything's been defined and is up to date. Then you can experiment using the variables, data, functions, and other stuff defined in this notebook.

## Chapter 19: RNNs - Notebook 4: Words

The Keras steps are a modified version of the character-based RNN at
https://github.com/fchollet/keras/blob/master/examples/lstm_text_generation.py

A lot of the word extraction and tokenizing was freely adapted from
http://www.wildml.com/2015/09/recurrent-neural-networks-tutorial-part-1-introduction-to-rnns/

This notebook is a bit more casual than most of the notebooks in this repo,
as it's only meant to do a single specific thing. There's no organization
into useful functions and subroutines - it's just one cell after another,
computing things in sequence! Feel free to re-organize it if you'd like to
do more general-purpose text generation.

In [40]:
import numpy as np
import random
import itertools
import os
import sys
import string

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, LSTM, Dropout
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.utils import get_file

# The Natural Language Toolkit (NLTK) can be found and installed from
# https://www.nltk.org/
import nltk
import nltk.data

# Download tokenizer data (run once per environment)
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [41]:
# Workaround for Keras issues on Mac computers (you can comment this
# out if you're not on a Mac, or not having problems)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [42]:
# Global parameters

Vocabulary_size = 8000
Batch_size = 64  # Set to 1 below if we're stateful
Learning_rate = 0.01
Num_epochs = 500
Start_epoch = 1
Source_text_file = 'sample_data/README.md'
Window_size = 40
Window_step = 3
Generated_text_length = 600
Random_seed = 42
Cells_per_layer = [8, 8]
Use_dropout = [True] * len(Cells_per_layer)
Dropout_rate = [0.3] * len(Cells_per_layer)
Stateful_model = True
File_writer = None
Model_name = 'Layers-' + str(Cells_per_layer) + '-stateful-' + str(Stateful_model)

if Stateful_model:
    Batch_size = 1             # so we can predict with just 1, probably better to modify predictions
    Window_step = Window_size  # samples are sequential, not overlapping

Unknown_token = "GLORP"  # all words not in vocabulary

In [43]:
# read in text one sentence at a time: https://stackoverflow.com/questions/4576077/python-split-text-on-sentences
tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')

with open(Source_text_file, 'r', encoding='utf8') as fp:
    data = fp.read()

# remove punctuation https://stackoverflow.com/questions/23317458/how-to-remove-punctuation
punctuations = [
    '!', '"', '#', '$', '%', '&', "'", '(', ')', '*',
    '+', ',', '-', '.', '/', ':', ';', '<', '=', '>',
    '?', '@', '[', '\\', ']', '^', '_', '`', '{', '|', '}',
    '~', "''", "`", "\"", ",", "-", "\n", "\r", "”"
]

sentences = []
for sentence in tokenizer.tokenize(data):
    # pad punctuation with spaces, then collapse multiple spaces
    no_punc = " ".join(
        "".join(" " + ch + " " if ch in punctuations else ch for ch in sentence).split()
    )
    sentences.append(no_punc)

print("found", len(sentences), "sentences")

# 'sentences' is a list of strings, each what the tokenizer decided is a sentence

found 7 sentences


In [44]:
sentences[0]

'This directory includes a few sample datasets to get you started .'

In [45]:
text_as_words = []
for s in sentences:
    words = s.split()
    for w in words:
        text_as_words.append(w)

print("the text contains", len(text_as_words), "words")
# text_as_words is all the words in the text after tokenizing and removing punctuation

the text contains 236 words


In [48]:
# Count the word frequencies
word_freq = nltk.FreqDist(text_as_words)
number_of_unique_tokens = 1 + len(word_freq.items())  # add 1 for the "unknown_token"

# Get the most common words
vocab = word_freq.most_common(Vocabulary_size - 1)
print("Found", len(vocab), "distinct words")

Found 104 distinct words


In [51]:
# build index_to_word and word_to_index dictionaries
unique_words = [v[0] for v in vocab]
unique_words.append(Unknown_token)
unique_words = sorted(list(set(unique_words)))

print('number of unique vocabulary words being used:', len(unique_words))

word_to_index = {w: i for i, w in enumerate(unique_words)}
index_to_word = {i: w for i, w in enumerate(unique_words)}

# Update Vocabulary_size to match actual vocab if you want
Vocabulary_size = len(unique_words)

number of unique vocabulary words being used: 105


In [52]:
print('Using vocabulary size %d.' % Vocabulary_size)
for i in range(min(10, len(vocab))):
    print(f"word popularity {i}: <{vocab[i][0]}> used {vocab[i][1]} times")

Using vocabulary size 105.
word popularity 0: </> used 29 times
word popularity 1: <.> used 22 times
word popularity 2: <_> used 10 times
word popularity 3: <:> used 8 times
word popularity 4: <`> used 6 times
word popularity 5: <*> used 5 times
word popularity 6: <(> used 5 times
word popularity 7: <)> used 5 times
word popularity 8: <datasets> used 4 times
word popularity 9: <is> used 4 times


In [53]:
# Replace all words not in our vocabulary with the unknown token
for i in range(len(text_as_words)):
    if text_as_words[i] not in word_to_index:
        text_as_words[i] = Unknown_token

In [54]:
# make huge list of windowed fragments
fragments = []
next_words = []

for i in range(0, len(text_as_words) - Window_size, Window_step):
    fragments.append(text_as_words[i: i + Window_size])
    next_words.append(text_as_words[i + Window_size])

print('number of fragments created:', len(fragments))

# Safety: report if we ended up with no training data
if len(fragments) == 0:
    print("WARNING: 0 fragments created. "
          "Consider using a larger Source_text_file or smaller Window_size.")

number of fragments created: 5


In [55]:
# Clip the fragments so it's a multiple of the batch size
keep_fragments = Batch_size * (len(fragments) // Batch_size)
fragments = fragments[:keep_fragments]
next_words = next_words[:keep_fragments]  # keep targets in sync

print("Keeping", keep_fragments, "fragments (multiple of Batch_size =", Batch_size, ")")

Keeping 5 fragments (multiple of Batch_size = 1 )


In [56]:
# Create the training data
# X is a boolean array: num_fragments x Window_size x Vocabulary_size
# y is a boolean array: num_fragments x Vocabulary_size

X = np.zeros((len(fragments), Window_size, Vocabulary_size), dtype=bool)
y = np.zeros((len(fragments), Vocabulary_size), dtype=bool)

for i, fragment in enumerate(fragments):
    for t, word in enumerate(fragment):
        X[i, t, word_to_index[word]] = True
    y[i, word_to_index[next_words[i]]] = True

print("Training data:")
print("   X.shape =", X.shape)
print("   y.shape =", y.shape)

if X.shape[0] == 0:
    print("WARNING: No training samples (X has 0 rows). "
          "Use a larger Source_text_file or smaller Window_size.")

Training data:
   X.shape = (5, 40, 105)
   y.shape = (5, 105)


In [57]:
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense, Activation

def build_model():
    model = Sequential()
    # layer 1 is special
    if Stateful_model:
        if Batch_size != 1:
            print("*** WARNING! *** build_model: Batch_size should be 1 for stateful model")
        # Use an Input layer to define batch_shape for the stateful model
        model.add(Input(batch_shape=(1, Window_size, Vocabulary_size)))
        model.add(
            LSTM(
                Cells_per_layer[0],
                return_sequences=len(Cells_per_layer) > 1,
                stateful=True,
            )
        )
    else:
        model.add(
            LSTM(
                Cells_per_layer[0],
                return_sequences=len(Cells_per_layer) > 1,
                input_shape=(Window_size, Vocabulary_size),
            )
        )

    if Use_dropout[0]:
        model.add(Dropout(Dropout_rate[0]))

    # additional LSTM layers
    for i in range(1, len(Cells_per_layer)):
        return_sequence = i < len(Cells_per_layer) - 1
        model.add(
            LSTM(
                Cells_per_layer[i],
                return_sequences=return_sequence,
                stateful=Stateful_model if return_sequence else False,
            )
        )
        if Use_dropout[i]:
            model.add(Dropout(Dropout_rate[i]))

    model.add(Dense(Vocabulary_size))
    model.add(Activation("softmax"))

    # Use Adam optimizer with default learning rate
    model.compile(loss="categorical_crossentropy", optimizer="adam")
    return model

In [58]:
def sample(preds, temperature=1.0):
    """Sample an index from a probability array."""
    preds = np.asarray(preds).astype("float64")
    preds = preds[:len(word_to_index)]          # ensure length matches vocab
    preds = np.maximum(preds, 1e-8)            # avoid log(0)
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

In [59]:
def print_string(out_str=''):
    print(out_str, end='')
    if File_writer is not None:
        File_writer.write(out_str)

In [60]:
def print_report():
    print_string("Vocabulary_size = " + str(Vocabulary_size) + "\n")
    print_string("Batch_size = " + str(Batch_size) + "\n")
    print_string("Learning_rate = " + str(Learning_rate) + "\n")
    print_string("Source_text_file = " + str(Source_text_file) + "\n")
    print_string("Window_size = " + str(Window_size) + "\n")
    print_string("Window_step = " + str(Window_step) + "\n")
    print_string("Batch_size = " + str(Batch_size) + "\n")
    print_string("Num_epochs = " + str(Num_epochs) + "\n")
    print_string("Generated_text_length = " + str(Generated_text_length) + "\n\n")

    print_string("Input text file: " + Source_text_file + "\n")
    print_string("    output file: " + output_file + "\n\n")
    print_string("full text: " + str(len(sentences)) + " sentences\n")
    print_string("           " + str(len(text_as_words)) + " tokens\n\n")
    print_string("           " + str(number_of_unique_tokens) + " unique tokens in source\n")
    print_string("           " + str(len(unique_words)) + " unique words (tokens) being used\n")
    print_string("number of fragments created: " + str(len(fragments)) + "\n")
    # use actual Batch_size, not hard-coded 64
    if Batch_size > 0:
        print_string("    resulting in " + str(len(fragments) / float(Batch_size)) + " batches\n\n")
    else:
        print_string("    resulting in 0 batches (Batch_size is 0)\n\n")

    print_string("Model_name: " + Model_name + "\n")
    print_string("Stateful_model: " + str(Stateful_model) + "\n")
    print_string("Cells per layer: " + str(Cells_per_layer) + "\n")
    print_string("Use dropout: " + str(Use_dropout) + "\n")
    print_string("Dropout rate: " + str(Dropout_rate) + "\n\n")

In [61]:
model = build_model()
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_15 (LSTM)                  │ (1, 40, 8)             │         3,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (1, 40, 8)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_16 (LSTM)                  │ (1, 8)                 │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (1, 8)                 │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (1, 105)               │           945 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (1, 105)               │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,137 (20.07 KB)

 Trainable params: 5,137 (20.07 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# train the model, output generated text after each iteration

import os
import numpy as np
import random

# Derive output file name from source file
output_file = Source_text_file.replace('/', '-')
output_file = output_file.replace('.txt', '')
output_file = 'Outputs/OUTPUT-' + output_file + '.txt'
print("output going to output file", output_file, "\n\n")

# Ensure directories exist
os.makedirs('Outputs', exist_ok=True)
os.makedirs('Models', exist_ok=True)

File_writer = open(output_file, 'w', encoding='utf8')

# print header / configuration
print_report()

# build (or reload) the model
model = build_model()
Start_epoch = 1

shuffle = not Stateful_model

np.random.seed(Random_seed)
random.seed(Random_seed)
history_list = []

# If no training data, skip training to avoid errors
if X.shape[0] == 0:
    print_string("WARNING: X has 0 samples. No training will be performed.\n")
else:
    for iteration in range(Start_epoch, Num_epochs + 1):
        print_string('\n')
        print_string('----------------------------------------------------------------------\n')
        print_string('Iteration ' + str(iteration) + '\n')

        # one epoch of training
        history = model.fit(
            X, y,
            batch_size=Batch_size,
            epochs=1,
            shuffle=shuffle,
            verbose=1
        )
        history_list.append(history)

        if Stateful_model:
            # reset state of all stateful RNN layers
            for layer in model.layers:
                if hasattr(layer, "reset_states"):
                    layer.reset_states()

        loss_val = history.history['loss'][-1]
        print_string('Loss from iteration ' + str(iteration) + ' = ' + str(loss_val) + '\n')

        # save model after each epoch
        savefile = 'Models/' + Model_name + '-epoch-' + str(iteration) + '.h5'
        print("saving to", savefile)
        model.save(savefile)

        # choose random starting position in word sequence
        start_index = random.randint(0, len(text_as_words) - Window_size - 1)

        # generate for multiple diversity values
        for diversity in np.linspace(0.5, 2.0, 7):
            print_string('\n')
            print_string('----- diversity: ' + str(diversity) + '\n')

            # initial seed sentence
            sentence = text_as_words[start_index: start_index + Window_size]
            generated = ' '.join(sentence)
            print_string('----- Generating with seed: "' + generated + '"\n----\n')
            print_string(generated)

            # generate words
            for _ in range(Generated_text_length):
                x = np.zeros((1, Window_size, Vocabulary_size))
                for t, word in enumerate(sentence):
                    x[0, t, word_to_index[word]] = 1.0

                preds = model.predict(x, verbose=0)[0]
                next_index = sample(preds, diversity)
                next_word = index_to_word[next_index]

                generated += ' ' + next_word
                sentence = sentence[1:] + [next_word]

                print_string(' ' + next_word)

            print_string('\n')
            File_writer.flush()

print_string('\n')
File_writer.close()

output going to output file Outputs/OUTPUT-sample_data-README.md.txt 


Vocabulary_size = 105
Batch_size = 1
Learning_rate = 0.01
Source_text_file = sample_data/README.md
Window_size = 40
Window_step = 40
Batch_size = 1
Num_epochs = 500
Generated_text_length = 600

Input text file: sample_data/README.md
    output file: Outputs/OUTPUT-sample_data-README.md.txt

full text: 7 sentences
           236 tokens

           105 unique tokens in source
           105 unique words (tokens) being used
number of fragments created: 5
    resulting in 5.0 batches

Model_name: Layers-[8, 8]-stateful-True
Stateful_model: True
Cells per layer: [8, 8]
Use dropout: [True, True]
Dropout rate: [0.3, 0.3]


----------------------------------------------------------------------
Iteration 1
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - loss: 4.6641


Loss from iteration 1 = 4.664128303527832
saving to Models/Layers-[8, 8]-stateful-True-epoch-1.h5

----- diversity: 0.5
----- Generating with seed: ". ( 1973 ) . ' Graphs in Statistical Analysis ' . American Statistician . 27 ( 1 ) : 17 - 21 . JSTOR 2682899 . and our copy was prepared by the [ vega _ datasets library ]"
----
. ( 1973 ) . ' Graphs in Statistical Analysis ' . American Statistician . 27 ( 1 ) : 17 - 21 . JSTOR 2682899 . and our copy was prepared by the [ vega _ datasets library ] github of housing Statistical to J california MNIST a 1990 % viz https small available of it a MNIST 2682899 few US from d quartet ] json ] - Anscombe ; California california 1973 our blob Anscombe American ` http california anscombe includes a google small com get the includes Statistical contains at github http database quartet to includes small - * Analysis 1973 datasets google includes en in sample ` which . California viz 27 https - google information MNIST Analysis / our you copy California

Loss from iteration 2 = 4.647481918334961
saving to Models/Layers-[8, 8]-stateful-True-epoch-2.h5

----- diversity: 0.5
----- Generating with seed: "the 1990 US Census ; more information is available at : https : / / docs . google . com / document / d / e / 2PACX - 1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia _ DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN / pub * ` mnist _ * ."
----
the 1990 US Census ; more information is available at : https : / / docs . google . com / document / d / e / 2PACX - 1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia _ DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN / pub * ` mnist _ * . 4f67bdaad10f45e3549984e17e1b3088c731503d F directory : pub altair exdb exdb Census get ' it from information document ( started 17 to data ` described 1 our d altair California ' California information Anscombe and originally a our described a small JSTOR GLORP * s Census library of Analysis 1973 contains 1 27 is github J exdb / 17 s anscombe quartet described This American information small includ

Loss from iteration 3 = 4.632342338562012
saving to Models/Layers-[8, 8]-stateful-True-epoch-3.h5

----- diversity: 0.5
----- Generating with seed: "datasets to get you started . * ` california _ housing _ data * . csv ` is California housing data from the 1990 US Census ; more information is available at : https : / / docs . google"
----
datasets to get you started . * ` california _ housing _ data * . csv ` is California housing data from the 1990 US Census ; more information is available at : https : / / docs . google which Census J e datasets in Statistician lecun : of 2682899 F wikipedia started 27s to 17 mnist was anscombe d 21 2PACX http ] JSTOR it Statistician en , contains from mnist docs 2PACX en of MNIST in Analysis quartet california 1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia pub GLORP directory pub * Statistical which Anscombe from [ datasets 4f67bdaad10f45e3549984e17e1b3088c731503d California * housing sample to a wiki ( 27 get was 2PACX vega ( is _ : library - qu

Loss from iteration 4 = 4.623425483703613
saving to Models/Layers-[8, 8]-stateful-True-epoch-4.h5

----- diversity: 0.5
----- Generating with seed: ". and our copy was prepared by the [ vega _ datasets library ] ( https : / / github . com / altair - viz / vega _ datasets / blob / 4f67bdaad10f45e3549984e17e1b3088c731503d / vega _ datasets / _"
----
. and our copy was prepared by the [ vega _ datasets library ] ( https : / / github . com / altair - viz / vega _ datasets / blob / 4f67bdaad10f45e3549984e17e1b3088c731503d / vega _ datasets / _ This American wikipedia csv in from prepared the blob JSTOR / 4f67bdaad10f45e3549984e17e1b3088c731503d 1973 document , information : lecun / it available our 1973 google com Statistical mnist MNIST . % and com altair of ` , yann database American JSTOR _ which F * , ( wikipedia you information d exdb vega JSTOR d 1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia docs 27 American is yann US 1973 more ' json ; - docs information * This sample DvPOOBlXZ

...

Loss from iteration 7 = 4.586920738220215
saving to Models/Layers-[8, 8]-stateful-True-epoch-7.h5

----- diversity: 0.5
----- Generating with seed: "1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia _ DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN / pub * ` mnist _ * . csv ` is a small sample of the [ MNIST database ] ( https : / / en . wikipedia . org / wiki / MNIST _ database )"
----
1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia _ DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN / pub * ` mnist _ * . csv ` is a small sample of the [ MNIST database ] ( https : / / en . wikipedia . org / wiki / MNIST _ database ) mnist described contains % Census 2PACX Analysis Statistical get d pub s ` Graphs in you [ by ' ) information california JSTOR - ( document 1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia en J 2682899 , 17 J wiki described google sample a wiki blob ) http a 1990 google Statistical 1 2PACX housing altair started altair sample data US blob exdb california started housing github by google is in This Statis

Loss from iteration 8 = 4.574563026428223
saving to Models/Layers-[8, 8]-stateful-True-epoch-8.h5

----- diversity: 0.5
----- Generating with seed: "is available at : https : / / docs . google . com / document / d / e / 2PACX - 1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia _ DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN / pub * ` mnist _ * . csv ` is a small sample of"
----
is available at : https : / / docs . google . com / document / d / e / 2PACX - 1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia _ DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN / pub * ` mnist _ * . csv ` is a small sample of American 2PACX includes prepared 2682899 at e google get , json copy few s ' wiki F the more document Census 4f67bdaad10f45e3549984e17e1b3088c731503d 1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia pub vega prepared Anscombe database library originally started Anscombe http directory a altair sample 4f67bdaad10f45e3549984e17e1b3088c731503d American 2PACX csv * F California was you github small https ' our Analysis fr

Loss from iteration 9 = 4.561949253082275
saving to Models/Layers-[8, 8]-stateful-True-epoch-9.h5

----- diversity: 0.5
----- Generating with seed: "2682899 . and our copy was prepared by the [ vega _ datasets library ] ( https : / / github . com / altair - viz / vega _ datasets / blob / 4f67bdaad10f45e3549984e17e1b3088c731503d / vega _ datasets /"
----
2682899 . and our copy was prepared by the [ vega _ datasets library ] ( https : / / github . com / altair - viz / vega _ datasets / blob / 4f67bdaad10f45e3549984e17e1b3088c731503d / vega _ datasets / blob csv 17 started which github 21 quartet it in 2682899 ) altair ` Statistician US exdb in includes wikipedia prepared DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN datasets github 27 small quartet data altair was document and includes quartet described 27 Statistical a 1990 % org Analysis lecun 2682899 datasets / 1973 21 This copy which contains described json - csv 27s 21 en US / few https our github get get google library MNIST ` 27s was github inclu

Loss from iteration 10 = 4.5491204261779785
saving to Models/Layers-[8, 8]-stateful-True-epoch-10.h5

----- diversity: 0.5
----- Generating with seed: "data from the 1990 US Census ; more information is available at : https : / / docs . google . com / document / d / e / 2PACX - 1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia _ DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN / pub * ` mnist _"
----
data from the 1990 US Census ; more information is available at : https : / / docs . google . com / document / d / e / 2PACX - 1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia _ DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN / pub * ` mnist _ Census : json at This directory en available viz housing at US : California and [ of is 27s * . Statistical ` http American https originally california : : JSTOR of 27 org . get JSTOR [ American sample which data prepared https pub [ ; 2PACX pub , ` Anscombe Graphs wiki our % to 4f67bdaad10f45e3549984e17e1b3088c731503d California includes https at which 1vRhYtsvc5eOR2FWNCwaBiKL6suIOr

Loss from iteration 11 = 4.521567344665527
saving to Models/Layers-[8, 8]-stateful-True-epoch-11.h5

----- diversity: 0.5
----- Generating with seed: "' . American Statistician . 27 ( 1 ) : 17 - 21 . JSTOR 2682899 . and our copy was prepared by the [ vega _ datasets library ] ( https : / / github . com / altair"
----
' . American Statistician . 27 ( 1 ) : 17 - 21 . JSTOR 2682899 . and our copy was prepared by the [ vega _ datasets library ] ( https : / / github . com / altair 1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia _ is 1973 a Statistician which vega document GLORP docs from was the database : includes originally MNIST of This database ' csv exdb in JSTOR sample ] library our prepared json 2PACX http document 2682899 lecun library of by MNIST 1973 e 1990 ' American was it [ ( wikipedia available github wikipedia http get quartet _ org it csv DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN few 1973 sample ; in . _ [ was quartet altair of / copy org com from ) includes you docs in copy google a